# NB20 · Intermediate Capstone

`intermediate/` · notebook 20 of 32 · 22 questions + a mini-project

This is the end of the intermediate tier. Every question here is a small
real program that mixes tools from NB09 to NB19: classes, custom
exceptions, files and JSON, regex, the standard library, and the data
structures from NB19.

Nobody tells you which tool to use any more. Before you type, spend a
minute deciding: what are the pieces, what should be a class or a
function, which errors can happen and who should handle them? **Design
decisions matter as much as correct output.** The explanations talk about
those decisions, not just about the code.

Questions start at L3. When a question asks you to write a class and then
use it, the code to run is shown in a `py` block: paste it below your class.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB09 to NB19: all intermediate notebooks

## What you'll practise

- Multi-part problems that use OOP, exceptions, files and the standard library together

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L3 · Intermediate

### NB20-Q01 · L3 · Classes + custom exceptions · Write

Write an `Inventory` class with a `stock` dict and two methods:

- `add(item, qty)`
- `remove(item, qty)`, which raises a custom `OutOfStock` exception when
  there isn't enough. Its message is `not enough <item> (have <n>, need <qty>)`.

Process every order in `orders`. Print what happened, catch `OutOfStock`
and print its message, then print the final stock.

```text
added 10 apple
removed 3 apple
error: not enough pear (have 0, need 1)
error: not enough apple (have 7, need 9)
added 4 pear
stock: {'apple': 7, 'pear': 4}
```

**Starting code (already in the cell below):**

```python
orders = [("add", "apple", 10), ("remove", "apple", 3), ("remove", "pear", 1),
          ("remove", "apple", 9), ("add", "pear", 4)]
```

<details><summary><b>Hint</b></summary>

`class OutOfStock(Exception): pass` is a complete custom exception. `remove` should check *before* changing anything, then `raise OutOfStock(...)`. The loop does the `try` / `except`.

</details>

<details><summary><b>Expected output</b></summary>

```text
added 10 apple
removed 3 apple
error: not enough pear (have 0, need 1)
error: not enough apple (have 7, need 9)
added 4 pear
stock: {'apple': 7, 'pear': 4}
```

</details>

<details><summary><b>Solution</b></summary>

```python
class OutOfStock(Exception):
    pass


class Inventory:
    def __init__(self):
        self.stock = {}

    def add(self, item, qty):
        self.stock[item] = self.stock.get(item, 0) + qty

    def remove(self, item, qty):
        have = self.stock.get(item, 0)
        if qty > have:
            raise OutOfStock(f"not enough {item} (have {have}, need {qty})")
        self.stock[item] = have - qty


inventory = Inventory()
for action, item, qty in orders:
    try:
        if action == "add":
            inventory.add(item, qty)
            print("added", qty, item)
        else:
            inventory.remove(item, qty)
            print("removed", qty, item)
    except OutOfStock as error:
        print("error:", error)
print("stock:", inventory.stock)
```

The class *detects* the problem and the caller *decides* what to do about it. That split is the point of raising an exception instead of printing inside `remove`: a web shop might show a message, a script might stop. Checking before changing `stock` means a failed order leaves nothing half-done. The `print` after each call only runs when the call didn't raise.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class OutOfStock(Exception):
    def __init__(self, item, have, need):
        super().__init__(f"not enough {item} (have {have}, need {need})")
        self.item, self.have, self.need = item, have, need


class Inventory:
    def __init__(self):
        self.stock = {}

    def add(self, item, qty):
        if qty <= 0:
            raise ValueError(f"quantity must be positive, got {qty}")
        self.stock[item] = self.stock.get(item, 0) + qty

    def remove(self, item, qty):
        have = self.stock.get(item, 0)
        if qty > have:
            raise OutOfStock(item, have, qty)
        self.stock[item] = have - qty


inventory = Inventory()
for action, item, qty in orders:
    try:
        if action == "add":
            inventory.add(item, qty)
            print(f"added {qty} {item}")
        else:
            inventory.remove(item, qty)
            print(f"removed {qty} {item}")
    except OutOfStock as error:
        print(f"error: {error}")
print(f"stock: {inventory.stock}")
```

The senior's exception carries the facts (`item`, `have`, `need`) as attributes, so a caller can offer "order the 7 we have?" without parsing the message. It also rejects nonsense input (`qty <= 0`) with the built-in `ValueError`: custom exceptions for business rules, built-in ones for bad arguments.

</details>

In [ ]:
orders = [("add", "apple", 10), ("remove", "apple", 3), ("remove", "pear", 1),
          ("remove", "apple", 9), ("add", "pear", 4)]

# your code here


### NB20-Q02 · L3 · try / else / finally in methods · Predict

`finally` runs even when a method returns. Predict the output.

**What does this print?**

```python
class Account:
    def __init__(self, balance):
        self.balance = balance
        self.log = []

    def withdraw(self, amount):
        try:
            if amount > self.balance:
                raise ValueError("insufficient")
            self.balance -= amount
        except ValueError:
            self.log.append(f"refused {amount}")
            return False
        else:
            self.log.append(f"paid {amount}")
            return True
        finally:
            self.log.append("checked")

account = Account(50)
print(account.withdraw(30), account.withdraw(30))
print(account.balance, account.log)
```

<details><summary><b>Hint</b></summary>

The first withdrawal fits in the balance; the second doesn't. In both cases, `finally` runs on the way out, after `else` or `except` has decided the return value.

</details>

<details><summary><b>Expected output</b></summary>

```text
True False
20 ['paid 30', 'checked', 'refused 30', 'checked']
```

</details>

<details><summary><b>Solution</b></summary>

First call: no error, so `else` logs `paid 30` and returns `True`, and on the way out `finally` logs `checked`. Second call: 30 > 20 raises, `except` logs `refused 30` and returns `False`, and `finally` still logs `checked`. The balance only dropped once: `20`.

</details>

<details><summary><b>Senior dev solution</b></summary>

`finally` is for clean-up that must always happen (closing files, releasing locks, writing an audit line). Never `return` from inside `finally`: it silently replaces the real return value and swallows exceptions (NB12).

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB20-Q03 · L3 · Dicts: changing while looping · Error

Remove every item that's out of stock. Which error does this raise?

**Which error does this raise?**

```python
stock = {"apple": 4, "pear": 0, "plum": 2, "fig": 0}
for item in stock:
    if stock[item] == 0:
        del stock[item]
print(stock)
```

<details><summary><b>Hint</b></summary>

The loop is walking through the dict's keys while the body deletes one of them.

</details>

<details><summary><b>Expected output</b></summary>

```text
RuntimeError
```

</details>

<details><summary><b>Solution</b></summary>

`RuntimeError`: a dict can't change size while you're looping over it, because the loop would lose its place. Python notices on the next step and stops. The fix is to loop over a copy (`list(stock)`) or, better, build a new dict.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
stock = {"apple": 4, "pear": 0, "plum": 2, "fig": 0}
stock = {item: qty for item, qty in stock.items() if qty > 0}
print(stock)
```

A dict comprehension builds the dict you want instead of editing the one you're reading. No mutation while iterating, no copy needed, and the condition reads as the rule: keep what's in stock.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB20-Q04 · L3 · CSV + grouping · Write

`path` points to a CSV file of test scores. Some scores are blank. For each
class, in alphabetical order, print the average of the scores that exist,
how many there were, and how many were missing. Delete the temporary folder
at the end with `folder.cleanup()`.

```text
art: average 74.5 from 2 scores, 1 missing
math: average 81.0 from 3 scores, 0 missing
```

**Starting code (already in the cell below):**

```python
import csv
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
path = Path(folder.name) / "grades.csv"
path.write_text("student,class,score\nAna,math,90\nBen,math,72\nAna,art,85\n"
                "Cara,math,81\nBen,art,64\nCara,art,\n", encoding="utf-8")
```

<details><summary><b>Hint</b></summary>

`csv.DictReader` gives each row as a dict. A blank cell comes back as `""`. Two `defaultdict`s (one of lists, one of ints) keep the scores and the missing counts per class.

</details>

<details><summary><b>Expected output</b></summary>

```text
art: average 74.5 from 2 scores, 1 missing
math: average 81.0 from 3 scores, 0 missing
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import defaultdict

scores = defaultdict(list)
missing = defaultdict(int)
with open(path, newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if row["score"] == "":
            missing[row["class"]] += 1
        else:
            scores[row["class"]].append(int(row["score"]))

for name in sorted(scores):
    average = sum(scores[name]) / len(scores[name])
    print(f"{name}: average {average:.1f} from {len(scores[name])} scores, "
          f"{missing[name]} missing")
folder.cleanup()
```

Every value from a CSV is text, so `int()` converts each score, and a blank is `""`, not `0`. Treating a blank as 0 would drag the art average down to 49.7, a silent wrong answer. `defaultdict` removes the "is this class new?" check, and `missing[name]` gives `0` for a class with no blanks. `newline=""` is what the `csv` docs ask for when opening the file.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import defaultdict
from statistics import mean

scores = defaultdict(list)
missing = defaultdict(int)
with path.open(newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if score := row["score"].strip():
            scores[row["class"]].append(int(score))
        else:
            missing[row["class"]] += 1

for name in sorted(scores.keys() | missing.keys()):
    values = scores[name]
    average = f"{mean(values):.1f}" if values else "n/a"
    print(f"{name}: average {average} from {len(values)} scores, {missing[name]} missing")
folder.cleanup()
```

The senior loops over every class seen in *either* dict, so a class where every score is blank still appears (as `n/a`) instead of vanishing. `.strip()` treats `" "` as blank too. The walrus (NB18) names the cleaned score in the `if`. `statistics.mean` (NB15) says what's computed.

</details>

In [ ]:
import csv
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
path = Path(folder.name) / "grades.csv"
path.write_text("student,class,score\nAna,math,90\nBen,math,72\nAna,art,85\n"
                "Cara,math,81\nBen,art,64\nCara,art,\n", encoding="utf-8")

# your code here


### NB20-Q05 · L3 · JSON round trip · Fix

This saves a stock dict to JSON and loads it back, then crashes on the
lookup. Fix it so it prints `lamp`, and still cleans up the folder.

**Buggy code (copied into the cell below):**

```python
import json
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
path = Path(folder.name) / "stock.json"

stock = {101: "lamp", 102: "desk"}
path.write_text(json.dumps(stock), encoding="utf-8")

loaded = json.loads(path.read_text(encoding="utf-8"))
print(loaded[101])
folder.cleanup()
```

<details><summary><b>Hint</b></summary>

Print `loaded` and look closely at the keys. What kinds of key does JSON allow?

</details>

<details><summary><b>Expected output</b></summary>

```text
lamp
```

</details>

<details><summary><b>Solution</b></summary>

```python
import json
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
path = Path(folder.name) / "stock.json"

stock = {101: "lamp", 102: "desk"}
path.write_text(json.dumps(stock), encoding="utf-8")

loaded = json.loads(path.read_text(encoding="utf-8"))
loaded = {int(key): value for key, value in loaded.items()}
print(loaded[101])
folder.cleanup()
```

JSON object keys are always strings, so `json.dumps` quietly turns `101` into `"101"`. After loading, `loaded[101]` looks for an `int` key that isn't there: `KeyError`. Converting the keys back right after loading restores the original shape. Tuples have the same problem: they come back as lists.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "stock.json"
    stock = {101: "lamp", 102: "desk"}
    records = [{"id": item_id, "name": name} for item_id, name in stock.items()]
    path.write_text(json.dumps(records), encoding="utf-8")

    loaded = {row["id"]: row["name"] for row in json.loads(path.read_text(encoding="utf-8"))}
    print(loaded[101])
```

Two senior habits. Store ids as *values* (a list of records), where JSON keeps their type, instead of as keys. And use `with TemporaryDirectory()`, which cleans up even when something inside raises. The original never reaches `folder.cleanup()` when it crashes.

</details>

In [ ]:
import json
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
path = Path(folder.name) / "stock.json"

stock = {101: "lamp", 102: "desk"}
path.write_text(json.dumps(stock), encoding="utf-8")

loaded = json.loads(path.read_text(encoding="utf-8"))
print(loaded[101])
folder.cleanup()


### NB20-Q06 · L3 · Dispatch table · Refactor

This to-do list works, but every new command means another `elif`, and the
list lives in a loose global. Refactor it into a `TodoList` class plus a
dict that maps each command name to a method. Keep the output identical.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
todo = []
commands = ["add milk", "add bread", "done milk", "list", "clear", "list",
            "add eggs", "list"]
for command in commands:
    parts = command.split(" ", 1)
    if parts[0] == "add":
        todo.append(parts[1])
        print("added " + parts[1])
    elif parts[0] == "done":
        todo.remove(parts[1])
        print("finished " + parts[1])
    elif parts[0] == "list":
        if len(todo) == 0:
            print("todo: (empty)")
        else:
            print("todo: " + ", ".join(todo))
    elif parts[0] == "clear":
        todo.clear()
        print("cleared")
```

<details><summary><b>Hint</b></summary>

Bound methods like `todo.add` are values you can store in a dict (NB09). `name, *args = command.split(" ", 1)` gives a list that's empty for `list` and `clear`; `handler(*args)` passes it on.

</details>

<details><summary><b>Expected output</b></summary>

```text
added milk
added bread
finished milk
todo: bread
cleared
todo: (empty)
added eggs
todo: eggs
```

</details>

<details><summary><b>Solution</b></summary>

```python
class TodoList:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)
        print("added", item)

    def done(self, item):
        self.items.remove(item)
        print("finished", item)

    def show(self):
        print("todo:", ", ".join(self.items) or "(empty)")

    def clear(self):
        self.items.clear()
        print("cleared")


todo = TodoList()
handlers = {"add": todo.add, "done": todo.done, "list": todo.show, "clear": todo.clear}
commands = ["add milk", "add bread", "done milk", "list", "clear", "list",
            "add eggs", "list"]
for command in commands:
    name, *args = command.split(" ", 1)
    handlers[name](*args)
```

Each command is now a small method with a name, and the loop is two lines that never change: adding a command means writing a method and one dict entry. `", ".join([])` is `""`, which is falsy, so `or "(empty)"` covers the empty case without an `if`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class TodoList:
    def __init__(self):
        self.items = []

    def run(self, command):
        match command.split(" ", 1):
            case ["add", item]:
                self.items.append(item)
                print("added", item)
            case ["done", item]:
                self.items.remove(item)
                print("finished", item)
            case ["list"]:
                print("todo:", ", ".join(self.items) or "(empty)")
            case ["clear"]:
                self.items.clear()
                print("cleared")
            case _:
                print("unknown command:", command)


todo = TodoList()
for command in ["add milk", "add bread", "done milk", "list", "clear", "list",
                "add eggs", "list"]:
    todo.run(command)
```

`match` with sequence patterns (NB18) checks the command name *and* the number of arguments in one place, and `case _` handles typos. A dispatch dict wins when commands are registered from elsewhere (plugins); `match` wins when the set is fixed and the shapes differ.

</details>

In [ ]:
todo = []
commands = ["add milk", "add bread", "done milk", "list", "clear", "list",
            "add eggs", "list"]
for command in commands:
    parts = command.split(" ", 1)
    if parts[0] == "add":
        todo.append(parts[1])
        print("added " + parts[1])
    elif parts[0] == "done":
        todo.remove(parts[1])
        print("finished " + parts[1])
    elif parts[0] == "list":
        if len(todo) == 0:
            print("todo: (empty)")
        else:
            print("todo: " + ", ".join(todo))
    elif parts[0] == "clear":
        todo.clear()
        print("cleared")


### NB20-Q07 · L3 · Regex + Counter · Write

Each access-log line should look like `IP METHOD PATH STATUS`, where METHOD
is `GET` or `POST` and STATUS is 3 digits. Count the valid lines and the
malformed ones, print how many requests got each status (lowest code
first), and the most requested path.

```text
requests: 5 (1 malformed)
200: 3
404: 2
top path: /home (3 requests)
```

**Starting code (already in the cell below):**

```python
import re
from collections import Counter

lines = [
    "10.0.0.1 GET /home 200",
    "10.0.0.2 GET /about 404",
    "10.0.0.1 POST /login 200",
    "garbage line",
    "10.0.0.3 GET /home 200",
    "10.0.0.2 GET /home 404",
]
```

<details><summary><b>Hint</b></summary>

Write one pattern with named groups and use `fullmatch`, so a line has to match from start to end. Two `Counter`s do the counting; `most_common(1)` gives the top path.

</details>

<details><summary><b>Expected output</b></summary>

```text
requests: 5 (1 malformed)
200: 3
404: 2
top path: /home (3 requests)
```

</details>

<details><summary><b>Solution</b></summary>

```python
pattern = re.compile(r"(?P<ip>[\d.]+) (?P<method>GET|POST) (?P<path>/\S*) (?P<status>\d{3})")
statuses = Counter()
paths = Counter()
malformed = 0
for line in lines:
    found = pattern.fullmatch(line)
    if found is None:
        malformed += 1
        continue
    statuses[found["status"]] += 1
    paths[found["path"]] += 1

print(f"requests: {sum(statuses.values())} ({malformed} malformed)")
for status in sorted(statuses):
    print(f"{status}: {statuses[status]}")
top_path, count = paths.most_common(1)[0]
print(f"top path: {top_path} ({count} requests)")
```

`fullmatch` rejects anything with extra or missing parts, so `garbage line` is counted as malformed instead of crashing the program. Named groups make `found["status"]` read like the log format. Statuses are 3-digit strings, so sorting them as text gives numeric order too.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
LOG_LINE = re.compile(r"""
    (?P<ip>[\d.]+) \s
    (?P<method>GET|POST) \s
    (?P<path>/\S*) \s
    (?P<status>\d{3})
""", re.VERBOSE)

parsed = [found.groupdict() for line in lines if (found := LOG_LINE.fullmatch(line))]
statuses = Counter(entry["status"] for entry in parsed)
top_path, count = Counter(entry["path"] for entry in parsed).most_common(1)[0]

print(f"requests: {len(parsed)} ({len(lines) - len(parsed)} malformed)")
for status, total in sorted(statuses.items()):
    print(f"{status}: {total}")
print(f"top path: {top_path} ({count} requests)")
```

`re.VERBOSE` (NB17) lets the pattern be laid out one field per line, which matters when someone has to change it next year. Parsing everything into dicts first, then counting, separates *reading* from *analysing*.

</details>

In [ ]:
import re
from collections import Counter

lines = [
    "10.0.0.1 GET /home 200",
    "10.0.0.2 GET /about 404",
    "10.0.0.1 POST /login 200",
    "garbage line",
    "10.0.0.3 GET /home 200",
    "10.0.0.2 GET /home 404",
]

# your code here


---

## L4 · Somewhat Difficult

### NB20-Q08 · L4 · LRU cache (OrderedDict) · Write

Write an `LRUCache` class: a cache that holds at most `capacity` items and,
when full, throws out the **least recently used** one. Use an
`OrderedDict` called `items`.

- `get(key)` returns the value (or `None` if missing), and counts as a use.
- `put(key, value)` stores the value, counts as a use, and evicts the
  oldest item if there are now too many.

Then run:

```py
cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
print(cache.get("a"))
cache.put("c", 3)
print(cache.get("b"))
cache.put("a", 10)
cache.put("d", 4)
print(cache.get("c"), cache.get("a"), cache.get("d"))
print(list(cache.items))
```

```text
1
None
None 10 4
['a', 'd']
```

<details><summary><b>Hint</b></summary>

An `OrderedDict` remembers order and has `move_to_end(key)` and `popitem(last=False)`. Keep the most recently used key at the end; the front is then always the one to evict.

</details>

<details><summary><b>Expected output</b></summary>

```text
1
None
None 10 4
['a', 'd']
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import OrderedDict


class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.items = OrderedDict()

    def get(self, key):
        if key not in self.items:
            return None
        self.items.move_to_end(key)
        return self.items[key]

    def put(self, key, value):
        self.items[key] = value
        self.items.move_to_end(key)
        if len(self.items) > self.capacity:
            self.items.popitem(last=False)


cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
print(cache.get("a"))
cache.put("c", 3)
print(cache.get("b"))
cache.put("a", 10)
cache.put("d", 4)
print(cache.get("c"), cache.get("a"), cache.get("d"))
print(list(cache.items))
```

The order of `items` *is* the usage history: front = least recent, end = most recent. Reading `a` moves it to the end, so adding `c` evicts `b`. Updating `a` to 10 is a use too, so `d` then evicts `c`. Every operation is O(1). The trap is `put` on an existing key: assigning alone doesn't move it, which is why `move_to_end` follows.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import OrderedDict

_MISSING = object()


class LRUCache:
    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity must be at least 1")
        self.capacity = capacity
        self.items = OrderedDict()

    def get(self, key, default=None):
        value = self.items.get(key, _MISSING)
        if value is _MISSING:
            return default
        self.items.move_to_end(key)
        return value

    def put(self, key, value):
        self.items[key] = value
        self.items.move_to_end(key)
        while len(self.items) > self.capacity:
            self.items.popitem(last=False)


cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
print(cache.get("a"))
cache.put("c", 3)
print(cache.get("b"))
cache.put("a", 10)
cache.put("d", 4)
print(cache.get("c"), cache.get("a"), cache.get("d"))
print(list(cache.items))
```

A sentinel object (NB18) means a cached `None` can be told apart from a miss: the beginner version can't tell "cached None" from "not cached". It validates `capacity` up front. For caching function results, `functools.lru_cache` (NB11) is this exact structure, already written.

</details>

In [ ]:
# your code here


### NB20-Q09 · L4 · Class design: Matrix · Write

Write a `Matrix` class that refuses bad input with **clear** `ValueError`
messages:

- `Matrix(rows)`: `a matrix needs at least one row and one column` if
  empty; `rows must all have the same length` if ragged.
- `size()` returns text like `2x3`.
- `add(other)` and `multiply(other)` return a new `Matrix`, or raise
  `cannot add 2x2 and 1x3` / `cannot multiply 1x3 by 2x2`.
- `str()` of a matrix shows one row per line, each number right-aligned
  in 3 characters, separated by a space.

Then run:

```py
a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
c = Matrix([[1, 0, 2]])
print("a + b:")
print(a.add(b))
print("a x b:")
print(a.multiply(b))
print("c x column:")
print(c.multiply(Matrix([[1], [2], [3]])))
for attempt in [lambda: a.add(c), lambda: c.multiply(a),
                lambda: Matrix([[1, 2], [3]]), lambda: Matrix([])]:
    try:
        attempt()
    except ValueError as error:
        print("error:", error)
```

```text
a + b:
  6   8
 10  12
a x b:
 19  22
 43  50
c x column:
  7
error: cannot add 2x2 and 1x3
error: cannot multiply 1x3 by 2x2
error: rows must all have the same length
error: a matrix needs at least one row and one column
```

<details><summary><b>Hint</b></summary>

Validate in `__init__`, so a broken `Matrix` can never exist. Cell `[i][j]` of a product is the sum of `row i of self` times `column j of other`, item by item.

</details>

<details><summary><b>Expected output</b></summary>

```text
a + b:
  6   8
 10  12
a x b:
 19  22
 43  50
c x column:
  7
error: cannot add 2x2 and 1x3
error: cannot multiply 1x3 by 2x2
error: rows must all have the same length
error: a matrix needs at least one row and one column
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Matrix:
    def __init__(self, rows):
        if not rows or not rows[0]:
            raise ValueError("a matrix needs at least one row and one column")
        for row in rows:
            if len(row) != len(rows[0]):
                raise ValueError("rows must all have the same length")
        self.rows = [list(row) for row in rows]
        self.height = len(rows)
        self.width = len(rows[0])

    def size(self):
        return f"{self.height}x{self.width}"

    def add(self, other):
        if self.height != other.height or self.width != other.width:
            raise ValueError(f"cannot add {self.size()} and {other.size()}")
        return Matrix([[a + b for a, b in zip(row, other_row)]
                       for row, other_row in zip(self.rows, other.rows)])

    def multiply(self, other):
        if self.width != other.height:
            raise ValueError(f"cannot multiply {self.size()} by {other.size()}")
        result = []
        for i in range(self.height):
            new_row = []
            for j in range(other.width):
                new_row.append(sum(self.rows[i][k] * other.rows[k][j]
                                   for k in range(self.width)))
            result.append(new_row)
        return Matrix(result)

    def __str__(self):
        return "\n".join(" ".join(f"{value:>3}" for value in row) for row in self.rows)


a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
c = Matrix([[1, 0, 2]])
print("a + b:")
print(a.add(b))
print("a x b:")
print(a.multiply(b))
print("c x column:")
print(c.multiply(Matrix([[1], [2], [3]])))
for attempt in [lambda: a.add(c), lambda: c.multiply(a),
                lambda: Matrix([[1, 2], [3]]), lambda: Matrix([])]:
    try:
        attempt()
    except ValueError as error:
        print("error:", error)
```

All checks happen *before* any work, and each message names both shapes, so whoever sees it knows exactly what went wrong. Validating in `__init__` means every other method can trust `self.rows`. Storing a *copy* of each row (`list(row)`) stops the caller's list from changing the matrix later. `ValueError` is the right built-in: the arguments have the right type but unusable values.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Matrix:
    def __init__(self, rows):
        if not rows or not rows[0]:
            raise ValueError("a matrix needs at least one row and one column")
        if any(len(row) != len(rows[0]) for row in rows):
            raise ValueError("rows must all have the same length")
        self.rows = tuple(tuple(row) for row in rows)

    @property
    def shape(self):
        return len(self.rows), len(self.rows[0])

    def size(self):
        return "{}x{}".format(*self.shape)

    def add(self, other):
        if self.shape != other.shape:
            raise ValueError(f"cannot add {self.size()} and {other.size()}")
        return Matrix([[a + b for a, b in zip(r1, r2)] for r1, r2 in zip(self.rows, other.rows)])

    def multiply(self, other):
        if self.shape[1] != other.shape[0]:
            raise ValueError(f"cannot multiply {self.size()} by {other.size()}")
        columns = list(zip(*other.rows))
        return Matrix([[sum(a * b for a, b in zip(row, col)) for col in columns]
                       for row in self.rows])

    __add__ = add
    __matmul__ = multiply

    def __str__(self):
        return "\n".join(" ".join(f"{value:>3}" for value in row) for row in self.rows)


a = Matrix([[1, 2], [3, 4]])
b = Matrix([[5, 6], [7, 8]])
c = Matrix([[1, 0, 2]])
print("a + b:")
print(a + b)
print("a x b:")
print(a @ b)
print("c x column:")
print(c @ Matrix([[1], [2], [3]]))
for attempt in [lambda: a + c, lambda: c @ a,
                lambda: Matrix([[1, 2], [3]]), lambda: Matrix([])]:
    try:
        attempt()
    except ValueError as error:
        print("error:", error)
```

Tuples make the matrix immutable, so nobody can break it after the checks. `zip(*other.rows)` (NB10) transposes, turning the product into "row times column". `__add__` and `__matmul__` (NB21) let users write `a + b` and `a @ b`, the same operator NumPy uses for matrix products.

</details>

In [ ]:
# your code here


### NB20-Q10 · L4 · __eq__ and hashing · Error

`Point` defines `__eq__` so equal coordinates compare equal. Then it goes
into a set. Which error does this raise?

**Which error does this raise?**

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

print(Point(1, 2) == Point(1, 2))
visited = {Point(1, 2)}
```

<details><summary><b>Hint</b></summary>

Sets and dict keys need a hash. What does Python do to `__hash__` when a class defines its own `__eq__`?

</details>

<details><summary><b>Expected output</b></summary>

```text
True
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError` (unhashable type): defining `__eq__` makes Python set `__hash__` to `None`. The reason is a rule: objects that are equal must have equal hashes, and the default hash (based on identity) would break it. So Python refuses to hash the object at all rather than let a set misbehave.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

print(Point(1, 2) == Point(1, 2))
visited = {Point(1, 2), Point(1, 2)}
print(len(visited))
```

Define `__hash__` from the same fields as `__eq__` (NB21), and only for objects you won't change after they go into a set. A grid search's `visited` set (NB19) usually just stores `(x, y)` tuples, which already hash correctly. `@dataclass(frozen=True)` (NB21) writes both methods for you.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB20-Q11 · L4 · Two stacks: undo / redo · Write

Write an `Editor` class with `text` (starting empty) and four methods:
`type(s)` appends text, `delete(n)` removes the last `n` characters,
`undo()` and `redo()`. Keep two stacks of earlier texts. `undo` / `redo`
with nothing to do raise a custom `EditorError` with the message
`nothing to undo` / `nothing to redo`.

**Any new edit clears the redo stack**, as in every real editor.

Run every action in `actions` and print one line each:

```text
type 'Hello' -> 'Hello'
type ' world' -> 'Hello world'
delete 6 -> 'Hello'
undo -> 'Hello world'
undo -> 'Hello'
redo -> 'Hello world'
type '!' -> 'Hello world!'
redo -> nothing to redo
undo -> 'Hello world'
undo -> 'Hello'
undo -> ''
undo -> nothing to undo
```

**Starting code (already in the cell below):**

```python
actions = [("type", "Hello"), ("type", " world"), ("delete", 6), ("undo", None),
           ("undo", None), ("redo", None), ("type", "!"), ("redo", None),
           ("undo", None), ("undo", None), ("undo", None), ("undo", None)]
```

<details><summary><b>Hint</b></summary>

Before every edit, push the current text onto the undo stack and empty the redo stack. `undo` pushes the current text onto *redo* and pops the previous one from *undo*; `redo` is the mirror image. Use `!r` in an f-string to print the quotes.

</details>

<details><summary><b>Expected output</b></summary>

```text
type 'Hello' -> 'Hello'
type ' world' -> 'Hello world'
delete 6 -> 'Hello'
undo -> 'Hello world'
undo -> 'Hello'
redo -> 'Hello world'
type '!' -> 'Hello world!'
redo -> nothing to redo
undo -> 'Hello world'
undo -> 'Hello'
undo -> ''
undo -> nothing to undo
```

</details>

<details><summary><b>Solution</b></summary>

```python
class EditorError(Exception):
    pass


class Editor:
    def __init__(self):
        self.text = ""
        self.undo_stack = []
        self.redo_stack = []

    def _save(self):
        self.undo_stack.append(self.text)
        self.redo_stack.clear()

    def type(self, s):
        self._save()
        self.text += s

    def delete(self, n):
        self._save()
        self.text = self.text[:len(self.text) - n]

    def undo(self):
        if not self.undo_stack:
            raise EditorError("nothing to undo")
        self.redo_stack.append(self.text)
        self.text = self.undo_stack.pop()

    def redo(self):
        if not self.redo_stack:
            raise EditorError("nothing to redo")
        self.undo_stack.append(self.text)
        self.text = self.redo_stack.pop()


editor = Editor()
for name, arg in actions:
    label = name if arg is None else f"{name} {arg!r}"
    try:
        if name == "type":
            editor.type(arg)
        elif name == "delete":
            editor.delete(arg)
        elif name == "undo":
            editor.undo()
        else:
            editor.redo()
        print(f"{label} -> {editor.text!r}")
    except EditorError as error:
        print(f"{label} -> {error}")
```

Every edit saves the old text first, so `undo` can step back to it, and `undo` saves the text it's leaving on the redo stack. Typing `!` after an undo starts a new history branch, so the old redo is thrown away; without `redo_stack.clear()`, redo would jump to a text that no longer follows from the current one. `self.text[:len(self.text) - n]` (rather than `[:-n]`) stays correct when `n` is `0`, since `[:-0]` would empty the text.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class EditorError(Exception):
    pass


class Editor:
    def __init__(self):
        self.text = ""
        self._undo, self._redo = [], []

    def _edit(self, new_text):
        self._undo.append(self.text)
        self._redo.clear()
        self.text = new_text

    def type(self, s):
        self._edit(self.text + s)

    def delete(self, n):
        self._edit(self.text[:max(len(self.text) - n, 0)])

    def _step(self, source, target, word):
        if not source:
            raise EditorError(f"nothing to {word}")
        target.append(self.text)
        self.text = source.pop()

    def undo(self):
        self._step(self._undo, self._redo, "undo")

    def redo(self):
        self._step(self._redo, self._undo, "redo")


editor = Editor()
for name, *args in actions:
    args = [a for a in args if a is not None]
    label = " ".join([name, *map(repr, args)])
    try:
        getattr(editor, name)(*args)
        print(f"{label} -> {editor.text!r}")
    except EditorError as error:
        print(f"{label} -> {error}")
```

`undo` and `redo` are the same move with the stacks swapped, so the senior writes it once (`_step`). Every edit goes through `_edit`, so the "clear redo" rule can't be forgotten by a future method. `getattr` (NB14) turns the action name into a method call, safe here because the names come from our own list, never from a user.

</details>

In [ ]:
actions = [("type", "Hello"), ("type", " world"), ("delete", 6), ("undo", None),
           ("undo", None), ("redo", None), ("type", "!"), ("redo", None),
           ("undo", None), ("undo", None), ("undo", None), ("undo", None)]

# your code here


### NB20-Q12 · L4 · Keeping data consistent · Fix

A transfer to a closed account fails, but money disappears: the total
should stay `150`. Fix `transfer` so a failed transfer changes nothing.

```text
transfer failed: account closed for Ben
Ana: 100 Ben: 50 total: 150
```

**Buggy code (copied into the cell below):**

```python
class AccountClosed(Exception):
    pass


class Account:
    def __init__(self, owner, balance, is_open=True):
        self.owner = owner
        self.balance = balance
        self.is_open = is_open

    def withdraw(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance -= amount

    def deposit(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance += amount


def transfer(source, target, amount):
    source.withdraw(amount)
    target.deposit(amount)


ana = Account("Ana", 100)
ben = Account("Ben", 50, is_open=False)
try:
    transfer(ana, ben, 30)
except AccountClosed as error:
    print("transfer failed: account closed for", error)
print("Ana:", ana.balance, "Ben:", ben.balance, "total:", ana.balance + ben.balance)
```

<details><summary><b>Hint</b></summary>

By the time `deposit` raises, `withdraw` has already happened. Either check both accounts before touching either, or undo the withdrawal when the deposit fails.

</details>

<details><summary><b>Expected output</b></summary>

```text
transfer failed: account closed for Ben
Ana: 100 Ben: 50 total: 150
```

</details>

<details><summary><b>Solution</b></summary>

```python
class AccountClosed(Exception):
    pass


class Account:
    def __init__(self, owner, balance, is_open=True):
        self.owner = owner
        self.balance = balance
        self.is_open = is_open

    def withdraw(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance -= amount

    def deposit(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance += amount


def transfer(source, target, amount):
    for account in (source, target):
        if not account.is_open:
            raise AccountClosed(account.owner)
    source.withdraw(amount)
    target.deposit(amount)


ana = Account("Ana", 100)
ben = Account("Ben", 50, is_open=False)
try:
    transfer(ana, ben, 30)
except AccountClosed as error:
    print("transfer failed: account closed for", error)
print("Ana:", ana.balance, "Ben:", ben.balance, "total:", ana.balance + ben.balance)
```

A transfer is two changes that must happen **together or not at all**. The buggy version raised halfway, after the withdrawal, so 30 vanished. Checking every condition before changing anything means an exception can only happen while nothing has changed yet. Each method alone was correct; the bug was in how they were combined.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class AccountClosed(Exception):
    pass


class Account:
    def __init__(self, owner, balance, is_open=True):
        self.owner = owner
        self.balance = balance
        self.is_open = is_open

    def withdraw(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance -= amount

    def deposit(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance += amount


def transfer(source, target, amount):
    source.withdraw(amount)
    try:
        target.deposit(amount)
    except Exception:
        source.deposit(amount)
        raise


ana = Account("Ana", 100)
ben = Account("Ben", 50, is_open=False)
try:
    transfer(ana, ben, 30)
except AccountClosed as error:
    print(f"transfer failed: account closed for {error}")
print(f"Ana: {ana.balance} Ben: {ben.balance} total: {ana.balance + ben.balance}")
```

Checking first only works if you can predict every failure. The senior version *compensates*: if the second step fails for **any** reason, undo the first, then re-raise with a bare `raise` so the caller still sees the real error. Databases give you this for free with transactions; NB24 shows a context manager that does the same.

</details>

In [ ]:
class AccountClosed(Exception):
    pass


class Account:
    def __init__(self, owner, balance, is_open=True):
        self.owner = owner
        self.balance = balance
        self.is_open = is_open

    def withdraw(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance -= amount

    def deposit(self, amount):
        if not self.is_open:
            raise AccountClosed(self.owner)
        self.balance += amount


def transfer(source, target, amount):
    source.withdraw(amount)
    target.deposit(amount)


ana = Account("Ana", 100)
ben = Account("Ben", 50, is_open=False)
try:
    transfer(ana, ben, 30)
except AccountClosed as error:
    print("transfer failed: account closed for", error)
print("Ana:", ana.balance, "Ben:", ben.balance, "total:", ana.balance + ben.balance)


### NB20-Q13 · L4 · Sliding window: rate limiter · Write

An API allows each user at most `LIMIT` requests in any `WINDOW`-second
period. A request at time `t` is allowed if fewer than `LIMIT` of that
user's **allowed** requests happened after `t - WINDOW`. Print each
decision, then how many were blocked.

```text
t=0 ana: allowed
t=1 ana: allowed
t=2 ben: allowed
t=3 ana: allowed
t=4 ana: blocked
t=11 ana: allowed
t=12 ana: allowed
t=12 ben: allowed
t=13 ana: allowed
blocked: 1
```

**Starting code (already in the cell below):**

```python
LIMIT = 3
WINDOW = 10
requests = [(0, "ana"), (1, "ana"), (2, "ben"), (3, "ana"), (4, "ana"),
            (11, "ana"), (12, "ana"), (12, "ben"), (13, "ana")]
```

<details><summary><b>Hint</b></summary>

Keep a `deque` of allowed request times per user (`defaultdict(deque)`). Before deciding, drop times from the left that are `<= t - WINDOW`: they've left the window.

</details>

<details><summary><b>Expected output</b></summary>

```text
t=0 ana: allowed
t=1 ana: allowed
t=2 ben: allowed
t=3 ana: allowed
t=4 ana: blocked
t=11 ana: allowed
t=12 ana: allowed
t=12 ben: allowed
t=13 ana: allowed
blocked: 1
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import defaultdict, deque

recent = defaultdict(deque)
blocked = 0
for t, user in requests:
    times = recent[user]
    while times and times[0] <= t - WINDOW:
        times.popleft()
    if len(times) < LIMIT:
        times.append(t)
        print(f"t={t} {user}: allowed")
    else:
        blocked += 1
        print(f"t={t} {user}: blocked")
print("blocked:", blocked)
```

Requests arrive in time order, so each user's deque is sorted: the oldest time is always on the left and can be dropped in O(1). At `t=4`, ana's window still holds 0, 1 and 3, so she's blocked. At `t=11`, 0 and 1 are 10 or more seconds old and drop out. Only *allowed* requests go into the deque, so being blocked doesn't make the block last longer.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import defaultdict, deque


class RateLimiter:
    def __init__(self, limit, window):
        self.limit = limit
        self.window = window
        self._recent = defaultdict(deque)

    def allow(self, user, t):
        times = self._recent[user]
        while times and times[0] <= t - self.window:
            times.popleft()
        if len(times) >= self.limit:
            return False
        times.append(t)
        return True


limiter = RateLimiter(LIMIT, WINDOW)
blocked = 0
for t, user in requests:
    ok = limiter.allow(user, t)
    blocked += not ok
    print(f"t={t} {user}: {'allowed' if ok else 'blocked'}")
print(f"blocked: {blocked}")
```

Wrapping the rule in a `RateLimiter` class separates the policy from the loop, so a web server could call `limiter.allow(user, now)` per request. `blocked += not ok` uses `bool` as `0`/`1` (NB01). Real limiters also forget users who've gone quiet, or the dict grows forever.

</details>

In [ ]:
LIMIT = 3
WINDOW = 10
requests = [(0, "ana"), (1, "ana"), (2, "ben"), (3, "ana"), (4, "ana"),
            (11, "ana"), (12, "ana"), (12, "ben"), (13, "ana")]

# your code here


### NB20-Q14 · L4 · Counter arithmetic · Predict

Comparing an order with a delivery. Predict the output.

**What does this print?**

```python
from collections import Counter

ordered = Counter(apple=5, pear=2, plum=3)
delivered = Counter(apple=5, pear=4, plum=1)

print(ordered - delivered)
print(delivered - ordered)
missing = ordered.copy()
missing.subtract(delivered)
print(missing)
```

<details><summary><b>Hint</b></summary>

The `-` operator keeps only counts that end up above zero. The `subtract()` method keeps every count, even zero and negative ones. A Counter prints its biggest counts first.

</details>

<details><summary><b>Expected output</b></summary>

```text
Counter({'plum': 2})
Counter({'pear': 2})
Counter({'plum': 2, 'apple': 0, 'pear': -2})
```

</details>

<details><summary><b>Solution</b></summary>

`ordered - delivered`: apple 0 and pear -2 are dropped, leaving `plum: 2` (two plums short). `delivered - ordered` leaves `pear: 2` (two extra pears). `subtract()` keeps everything: `plum: 2, apple: 0, pear: -2`, listed from biggest count to smallest. Pick the tool for the question: "what's missing?" wants `-`; "show me every difference" wants `subtract`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Silently dropping zero and negative counts is right for "what do we still need?" and wrong for an audit. Seniors pick the operation on purpose, and often print both directions, as here.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB20-Q15 · L4 · datetime: merging intervals · Write

Merge the room bookings that overlap **or touch**, and print the merged
blocks in time order. Then print the total booked time as `h:mm`.

```text
09:00-11:30
13:00-14:00
15:00-15:45
total booked: 4:15
```

**Starting code (already in the cell below):**

```python
from datetime import datetime

bookings = [("13:00", "14:00"), ("09:00", "10:30"), ("10:15", "11:00"),
            ("11:00", "11:30"), ("15:00", "15:45"), ("13:30", "13:45")]
```

<details><summary><b>Hint</b></summary>

Parse with `datetime.strptime(text, "%H:%M")` and sort by start time. Walk through: if a booking starts before (or exactly when) the current block ends, stretch the block. Careful: a booking can sit entirely inside another one.

</details>

<details><summary><b>Expected output</b></summary>

```text
09:00-11:30
13:00-14:00
15:00-15:45
total booked: 4:15
```

</details>

<details><summary><b>Solution</b></summary>

```python
parsed = []
for start, end in bookings:
    parsed.append((datetime.strptime(start, "%H:%M"), datetime.strptime(end, "%H:%M")))
parsed.sort()

merged = [list(parsed[0])]
for start, end in parsed[1:]:
    if start <= merged[-1][1]:
        merged[-1][1] = max(merged[-1][1], end)
    else:
        merged.append([start, end])

total_minutes = 0
for start, end in merged:
    print(f"{start:%H:%M}-{end:%H:%M}")
    total_minutes += int((end - start).total_seconds()) // 60
hours, minutes = divmod(total_minutes, 60)
print(f"total booked: {hours}:{minutes:02d}")
```

After sorting by start, any booking that overlaps the current block must start before that block ends, so one pass is enough: O(n log n) for the sort. `<=` merges `10:15-11:00` and `11:00-11:30`, which touch. `max()` is the trap: `13:30-13:45` sits inside `13:00-14:00`, and plain assignment would *shrink* the block to end at 13:45. Subtracting datetimes gives a `timedelta`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from datetime import timedelta

def parse(text):
    return datetime.strptime(text, "%H:%M")

merged = []
for start, end in sorted((parse(s), parse(e)) for s, e in bookings):
    if merged and start <= merged[-1][1]:
        merged[-1][1] = max(merged[-1][1], end)
    else:
        merged.append([start, end])

total = sum((end - start for start, end in merged), timedelta())
for start, end in merged:
    print(f"{start:%H:%M}-{end:%H:%M}")
hours, minutes = divmod(total // timedelta(minutes=1), 60)
print(f"total booked: {hours}:{minutes:02d}")
```

`if merged and ...` handles an empty booking list, where the beginner's `parsed[0]` would raise `IndexError`. `sum(..., timedelta())` adds durations directly, and `total // timedelta(minutes=1)` asks "how many whole minutes?" without going through seconds.

</details>

In [ ]:
from datetime import datetime

bookings = [("13:00", "14:00"), ("09:00", "10:30"), ("10:15", "11:00"),
            ("11:00", "11:30"), ("15:00", "15:45"), ("13:30", "13:45")]

# your code here


### NB20-Q16 · L4 · itertools.groupby · Fix

This should print one total per region, alphabetically:

```text
east 70
north 170
south 110
```

Instead, regions appear more than once. Fix it, still using `groupby`.

**Buggy code (copied into the cell below):**

```python
from itertools import groupby

sales = [("north", 120), ("south", 80), ("north", 50), ("east", 70), ("south", 30)]
for region, rows in groupby(sales, key=lambda sale: sale[0]):
    total = sum(amount for name, amount in rows)
    print(region, total)
```

<details><summary><b>Hint</b></summary>

`groupby` only groups items that sit **next to each other**. What has to happen to `sales` first?

</details>

<details><summary><b>Expected output</b></summary>

```text
east 70
north 170
south 110
```

</details>

<details><summary><b>Solution</b></summary>

```python
from itertools import groupby

sales = [("north", 120), ("south", 80), ("north", 50), ("east", 70), ("south", 30)]
by_region = sorted(sales, key=lambda sale: sale[0])
for region, rows in groupby(by_region, key=lambda sale: sale[0]):
    total = sum(amount for name, amount in rows)
    print(region, total)
```

`groupby` starts a new group every time the key *changes*, like folding runs in a list. On unsorted data `north` appears, then `south`, then `north` again, so you get five groups. Sorting by the same key first puts equal regions side by side, and it also makes the output alphabetical.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import Counter

sales = [("north", 120), ("south", 80), ("north", 50), ("east", 70), ("south", 30)]
totals = Counter()
for region, amount in sales:
    totals[region] += amount
for region in sorted(totals):
    print(region, totals[region])
```

For totals, a `Counter` (or `defaultdict(int)`) is one O(n) pass and doesn't care about order; sorting first costs O(n log n). Seniors use `groupby` when the data is *already* sorted, such as log lines by time or a sorted file too big to load.

</details>

In [ ]:
from itertools import groupby

sales = [("north", 120), ("south", 80), ("north", 50), ("east", 70), ("south", 30)]
for region, rows in groupby(sales, key=lambda sale: sale[0]):
    total = sum(amount for name, amount in rows)
    print(region, total)


---

## L5 · Hard

### NB20-Q17 · L5 · Shunting-yard calculator · Write

Build a calculator in three steps:

1. **Tokenise** each expression with a regex into numbers, operators and
   brackets.
2. Convert the tokens to **Reverse Polish Notation** (RPN) with the
   shunting-yard algorithm. `*` and `/` bind tighter than `+` and `-`, and
   all four are left-associative.
3. **Evaluate** the RPN with a stack.

Unbalanced brackets raise a custom `ExpressionError("mismatched brackets")`.

```text
3 + 4 * 2
  RPN: 3 4 2 * +
  value: 11
8 - 3 - 2
  RPN: 8 3 - 2 -
  value: 3
(1 + 2) * (3 + 4) - 5
  RPN: 1 2 + 3 4 + * 5 -
  value: 16
10 / 4 - 1
  RPN: 10 4 / 1 -
  value: 1.5
2 * (3 + 4
  error: mismatched brackets
1 + 2)
  error: mismatched brackets
```

**Starting code (already in the cell below):**

```python
import re

expressions = ["3 + 4 * 2", "8 - 3 - 2", "(1 + 2) * (3 + 4) - 5", "10 / 4 - 1",
               "2 * (3 + 4", "1 + 2)"]
```

<details><summary><b>Hint</b></summary>

Shunting-yard: numbers go straight to the output. Before pushing an operator, pop operators from the stack to the output while the top has **greater or equal** precedence (equal is what makes `8 - 3 - 2` left-associative). `(` is pushed; `)` pops until the matching `(`. To evaluate, pop the *right* operand first.

</details>

<details><summary><b>Expected output</b></summary>

```text
3 + 4 * 2
  RPN: 3 4 2 * +
  value: 11
8 - 3 - 2
  RPN: 8 3 - 2 -
  value: 3
(1 + 2) * (3 + 4) - 5
  RPN: 1 2 + 3 4 + * 5 -
  value: 16
10 / 4 - 1
  RPN: 10 4 / 1 -
  value: 1.5
2 * (3 + 4
  error: mismatched brackets
1 + 2)
  error: mismatched brackets
```

</details>

<details><summary><b>Solution</b></summary>

```python
class ExpressionError(Exception):
    pass


PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2}


def tokenise(text):
    return re.findall(r"\d+|[-+*/()]", text)


def to_rpn(tokens):
    output = []
    operators = []
    for token in tokens:
        if token.isdigit():
            output.append(token)
        elif token in PRECEDENCE:
            while (operators and operators[-1] in PRECEDENCE
                   and PRECEDENCE[operators[-1]] >= PRECEDENCE[token]):
                output.append(operators.pop())
            operators.append(token)
        elif token == "(":
            operators.append(token)
        else:
            while operators and operators[-1] != "(":
                output.append(operators.pop())
            if not operators:
                raise ExpressionError("mismatched brackets")
            operators.pop()
    while operators:
        operator = operators.pop()
        if operator == "(":
            raise ExpressionError("mismatched brackets")
        output.append(operator)
    return output


def evaluate(rpn):
    stack = []
    for token in rpn:
        if token.isdigit():
            stack.append(int(token))
            continue
        right = stack.pop()
        left = stack.pop()
        if token == "+":
            stack.append(left + right)
        elif token == "-":
            stack.append(left - right)
        elif token == "*":
            stack.append(left * right)
        else:
            stack.append(left / right)
    return stack[0]


for text in expressions:
    print(text)
    try:
        rpn = to_rpn(tokenise(text))
    except ExpressionError as error:
        print("  error:", error)
        continue
    print("  RPN:", " ".join(rpn))
    print("  value:", evaluate(rpn))
```

RPN puts each operator *after* its operands, so evaluation needs no precedence rules: push numbers, and each operator pops two and pushes the result. Shunting-yard handles precedence on the way in. Popping on *equal* precedence makes `8 - 3 - 2` mean `(8 - 3) - 2 = 3`; with `>` you'd get `8 - (3 - 2) = 7`. A `)` with no `(` on the stack, or a `(` left over at the end, means the brackets don't match. Splitting into three functions lets you test each step on its own.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import operator

class ExpressionError(Exception):
    pass

OPERATORS = {"+": (1, operator.add), "-": (1, operator.sub),
             "*": (2, operator.mul), "/": (2, operator.truediv)}
TOKEN = re.compile(r"\s*(?:(\d+)|(.))")

def tokenise(text):
    tokens = []
    for number, symbol in TOKEN.findall(text.strip()):
        if symbol and symbol not in OPERATORS and symbol not in "()":
            raise ExpressionError(f"unexpected character {symbol!r}")
        tokens.append(number or symbol)
    return tokens

def to_rpn(tokens):
    output, stack = [], []
    for token in tokens:
        match token:
            case "(":
                stack.append(token)
            case ")":
                while stack and stack[-1] != "(":
                    output.append(stack.pop())
                if not stack:
                    raise ExpressionError("mismatched brackets")
                stack.pop()
            case op if op in OPERATORS:
                while stack and stack[-1] in OPERATORS and OPERATORS[stack[-1]][0] >= OPERATORS[op][0]:
                    output.append(stack.pop())
                stack.append(op)
            case number:
                output.append(number)
    if "(" in stack:
        raise ExpressionError("mismatched brackets")
    return output + stack[::-1]

def evaluate(rpn):
    stack = []
    for token in rpn:
        if token in OPERATORS:
            right, left = stack.pop(), stack.pop()
            stack.append(OPERATORS[token][1](left, right))
        else:
            stack.append(int(token))
    return stack.pop()

for text in expressions:
    print(text)
    try:
        rpn = to_rpn(tokenise(text))
    except ExpressionError as error:
        print(f"  error: {error}")
        continue
    print(f"  RPN: {' '.join(rpn)}")
    print(f"  value: {evaluate(rpn)}")
```

One table holds both the precedence and the function for each operator (`operator.add`, NB10), so adding `%` is one line. The tokeniser rejects characters it doesn't know instead of silently skipping them, which the beginner `findall` does. Never use `eval()` (NB25) for a calculator: it runs *any* Python a user types.

</details>

In [ ]:
import re

expressions = ["3 + 4 * 2", "8 - 3 - 2", "(1 + 2) * (3 + 4) - 5", "10 / 4 - 1",
               "2 * (3 + 4", "1 + 2)"]

# your code here


### NB20-Q18 · L5 · Recursion + cycle detection · Write

Write a tiny spreadsheet. A cell holds a number like `"5"`, or a formula
like `"=A1+A2+1"` (cell names and numbers joined by `+`). Write a `Sheet`
class whose `value(name)` works out a cell, recursively, and **remembers**
every value it computes.

Raise custom exceptions that share a base class `SheetError`:

- `CycleError("cycle B1 -> B2 -> B1")` when a cell depends on itself
- `UnknownCell("unknown cell Z9")` for a reference to a missing cell

Print every cell in alphabetical order:

```text
A1 = 5
A2 = 10
A3 = 16
B1 = error: cycle B1 -> B2 -> B1
B2 = error: cycle B2 -> B1 -> B2
C1 = error: unknown cell Z9
C2 = error: unknown cell Z9
```

**Starting code (already in the cell below):**

```python
cells = {
    "A1": "5",
    "A2": "=A1+A1",
    "A3": "=A1+A2+1",
    "B1": "=B2+1",
    "B2": "=B1+1",
    "C1": "=A3+Z9",
    "C2": "=C1+1",
}
```

<details><summary><b>Hint</b></summary>

Pass the list of cells currently being worked out (the `path`) down the recursion. Meeting a cell that's already on the path means a cycle, and `path + [name]` is the message. Only store values that worked out.

</details>

<details><summary><b>Expected output</b></summary>

```text
A1 = 5
A2 = 10
A3 = 16
B1 = error: cycle B1 -> B2 -> B1
B2 = error: cycle B2 -> B1 -> B2
C1 = error: unknown cell Z9
C2 = error: unknown cell Z9
```

</details>

<details><summary><b>Solution</b></summary>

```python
class SheetError(Exception):
    pass


class CycleError(SheetError):
    pass


class UnknownCell(SheetError):
    pass


class Sheet:
    def __init__(self, cells):
        self.cells = cells
        self.values = {}

    def value(self, name, path=None):
        if path is None:
            path = []
        if name in self.values:
            return self.values[name]
        if name not in self.cells:
            raise UnknownCell(f"unknown cell {name}")
        if name in path:
            raise CycleError("cycle " + " -> ".join(path + [name]))
        raw = self.cells[name]
        if raw.startswith("="):
            total = 0
            for term in raw[1:].split("+"):
                if term.isdigit():
                    total += int(term)
                else:
                    total += self.value(term, path + [name])
        else:
            total = int(raw)
        self.values[name] = total
        return total


sheet = Sheet(cells)
for name in sorted(cells):
    try:
        print(f"{name} = {sheet.value(name)}")
    except SheetError as error:
        print(f"{name} = error: {error}")
```

Each formula asks for the cells it needs, which ask for theirs: recursion over a dependency graph (NB19). The `path` is the chain of cells still being worked out; meeting one of them again means the chain loops back, so it's a cycle, and the path *is* the error message. Remembering values makes each cell compute once. Both errors share `SheetError`, so one `except` handles every spreadsheet problem but nothing else. `path=None` avoids a shared mutable default.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class SheetError(Exception):
    pass


class CycleError(SheetError):
    pass


class UnknownCell(SheetError):
    pass


class Sheet:
    def __init__(self, cells):
        self.cells = dict(cells)
        self._values = {}

    def value(self, name):
        return self._value(name, ())

    def _value(self, name, path):
        if name in self._values:
            return self._values[name]
        if name not in self.cells:
            raise UnknownCell(f"unknown cell {name}")
        if name in path:
            raise CycleError("cycle " + " -> ".join((*path, name)))
        raw = self.cells[name]
        if not raw.startswith("="):
            result = int(raw)
        else:
            result = sum(int(term) if term.isdigit() else self._value(term, (*path, name))
                         for term in raw[1:].split("+"))
        self._values[name] = result
        return result


sheet = Sheet(cells)
for name in sorted(cells):
    try:
        print(f"{name} = {sheet.value(name)}")
    except SheetError as error:
        print(f"{name} = error: {error}")
```

The public `value(name)` hides the `path` parameter, so callers can't pass a wrong one, and an immutable tuple path can't be changed by accident. A real spreadsheet also has to *forget* stored values when a cell changes; that's why seniors think about cache invalidation before adding a cache. `graphlib` (NB19) could find cycles up front for the whole sheet.

</details>

In [ ]:
cells = {
    "A1": "5",
    "A2": "=A1+A1",
    "A3": "=A1+A2+1",
    "B1": "=B2+1",
    "B2": "=B1+1",
    "C1": "=A3+Z9",
    "C2": "=C1+1",
}

# your code here


### NB20-Q19 · L5 · Refactor to a class + exceptions · Refactor

This banking script works, but uses global dicts, repeats the same checks
in every function, and handles errors by printing deep inside. Refactor it
into a `Bank` class whose methods **raise** a custom `BankError`, with the
printing done by the caller. The output must not change.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
accounts = {"ana": 100, "ben": 20}
history = []

def deposit(name, amount):
    if name not in accounts:
        print("error: no account", name)
        return
    if amount <= 0:
        print("error: bad amount", amount)
        return
    accounts[name] = accounts[name] + amount
    history.append(name + " +" + str(amount))

def withdraw(name, amount):
    if name not in accounts:
        print("error: no account", name)
        return
    if amount <= 0:
        print("error: bad amount", amount)
        return
    if accounts[name] < amount:
        print("error: insufficient funds for", name)
        return
    accounts[name] = accounts[name] - amount
    history.append(name + " -" + str(amount))

deposit("ana", 50)
withdraw("ben", 30)
withdraw("ana", 120)
deposit("cara", 10)
deposit("ben", -5)
print(accounts)
print(history)
```

<details><summary><b>Hint</b></summary>

Move the two shared checks into one helper method that raises. Store the operations as `(method, name, amount)` tuples and run them in one loop with a single `try` / `except BankError`.

</details>

<details><summary><b>Expected output</b></summary>

```text
error: insufficient funds for ben
error: no account cara
error: bad amount -5
{'ana': 30, 'ben': 20}
['ana +50', 'ana -120']
```

</details>

<details><summary><b>Solution</b></summary>

```python
class BankError(Exception):
    pass


class Bank:
    def __init__(self, balances):
        self.balances = dict(balances)
        self.history = []

    def _check(self, name, amount):
        if name not in self.balances:
            raise BankError(f"no account {name}")
        if amount <= 0:
            raise BankError(f"bad amount {amount}")

    def deposit(self, name, amount):
        self._check(name, amount)
        self.balances[name] += amount
        self.history.append(f"{name} +{amount}")

    def withdraw(self, name, amount):
        self._check(name, amount)
        if self.balances[name] < amount:
            raise BankError(f"insufficient funds for {name}")
        self.balances[name] -= amount
        self.history.append(f"{name} -{amount}")


bank = Bank({"ana": 100, "ben": 20})
operations = [(bank.deposit, "ana", 50), (bank.withdraw, "ben", 30),
              (bank.withdraw, "ana", 120), (bank.deposit, "cara", 10),
              (bank.deposit, "ben", -5)]
for action, name, amount in operations:
    try:
        action(name, amount)
    except BankError as error:
        print("error:", error)
print(bank.balances)
print(bank.history)
```

The data and the rules now live together in one object, and the checks are written once. The bigger change is *who handles errors*: the methods only raise, and the caller decides to print. The same `Bank` could now sit behind a web page that shows errors in red, without changing a line of it. `dict(balances)` copies the starting balances so the bank owns its data.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class BankError(Exception):
    pass


class UnknownAccount(BankError):
    pass


class InsufficientFunds(BankError):
    pass


class Bank:
    def __init__(self, balances):
        self.balances = dict(balances)
        self.history = []

    def _check(self, name, amount):
        if name not in self.balances:
            raise UnknownAccount(f"no account {name}")
        if amount <= 0:
            raise BankError(f"bad amount {amount}")

    def _apply(self, name, change):
        self.balances[name] += change
        self.history.append(f"{name} {change:+}")

    def deposit(self, name, amount):
        self._check(name, amount)
        self._apply(name, amount)

    def withdraw(self, name, amount):
        self._check(name, amount)
        if self.balances[name] < amount:
            raise InsufficientFunds(f"insufficient funds for {name}")
        self._apply(name, -amount)


bank = Bank({"ana": 100, "ben": 20})
for action, name, amount in [("deposit", "ana", 50), ("withdraw", "ben", 30),
                             ("withdraw", "ana", 120), ("deposit", "cara", 10),
                             ("deposit", "ben", -5)]:
    try:
        getattr(bank, action)(name, amount)
    except BankError as error:
        print(f"error: {error}")
print(bank.balances)
print(bank.history)
```

Subclasses (`UnknownAccount`, `InsufficientFunds`) let a caller react differently to each problem while `except BankError` still catches them all. `f"{change:+}"` (NB02) prints the sign for you, so the history line comes from one place. Real money would use `Decimal` (NB16).

</details>

In [ ]:
accounts = {"ana": 100, "ben": 20}
history = []

def deposit(name, amount):
    if name not in accounts:
        print("error: no account", name)
        return
    if amount <= 0:
        print("error: bad amount", amount)
        return
    accounts[name] = accounts[name] + amount
    history.append(name + " +" + str(amount))

def withdraw(name, amount):
    if name not in accounts:
        print("error: no account", name)
        return
    if amount <= 0:
        print("error: bad amount", amount)
        return
    if accounts[name] < amount:
        print("error: insufficient funds for", name)
        return
    accounts[name] = accounts[name] - amount
    history.append(name + " -" + str(amount))

deposit("ana", 50)
withdraw("ben", 30)
withdraw("ana", 120)
deposit("cara", 10)
deposit("ben", -5)
print(accounts)
print(history)


### NB20-Q20 · L5 · Shared state traps · Predict

Three teams, two hidden traps. Predict **both** lines.

**What does this print?**

```python
class Team:
    members = []

    def __init__(self, name, roster=[]):
        self.name = name
        self.roster = roster

    def join(self, person):
        self.roster.append(person)
        Team.members.append(person)

red = Team("red")
blue = Team("blue")
green = Team("green", ["Zoe"])
red.join("Ann")
blue.join("Bob")
green.join("Cy")
print(red.roster, blue.roster, green.roster)
print(len(Team.members), red.roster is blue.roster)
```

<details><summary><b>Hint</b></summary>

When is the default `[]` created: once, or on every call? And how many `members` lists exist: one per team, or one for the class?

</details>

<details><summary><b>Expected output</b></summary>

```text
['Ann', 'Bob'] ['Ann', 'Bob'] ['Zoe', 'Cy']
3 True
```

</details>

<details><summary><b>Solution</b></summary>

The default `roster=[]` is created **once**, when `def` runs, so `red` and `blue` share that same list: both show `['Ann', 'Bob']`, and `is` is `True`. `green` passed its own list, so it gets `['Zoe', 'Cy']`. `members` is a class attribute, one list shared by every team, holding all 3 joins. Two separate traps (NB09 and NB14) that look identical in the output.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Team:
    def __init__(self, name, roster=None):
        self.name = name
        self.roster = list(roster) if roster is not None else []

    def join(self, person):
        self.roster.append(person)

red = Team("red")
blue = Team("blue")
green = Team("green", ["Zoe"])
red.join("Ann")
blue.join("Bob")
green.join("Cy")
print(red.roster, blue.roster, green.roster)
print(red.roster is blue.roster)
```

`roster=None` plus a fresh list per instance is the standard fix, and `list(roster)` copies the caller's list so the team doesn't share it either. If you really need a list of everyone, keep it in a separate `League` object rather than a hidden class attribute.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB20-Q21 · L5 · CSV + Decimal + JSON · Write

`ledger` is a CSV of `account,amount` lines. Read it, using `Decimal` for
money. For a bad line, raise a custom `LedgerError` inside a `parse_row`
function, catch it in the loop, and report it with its **line number in
the file** (the header is line 1). Then:

1. print each account's total (alphabetically) and the net total,
2. write a `summary.json` next to the CSV with the totals **as strings**,
3. read it back and print how many accounts it holds,
4. clean up the folder.

```text
line 5: bad amount 'abc'
line 7: expected 2 fields, got 1
food: -57.40
fun: -19.99
rent: -850.00
salary: 2400.50
net: 1473.11
summary.json holds 4 accounts
```

**Starting code (already in the cell below):**

```python
import csv
import json
import tempfile
from decimal import Decimal, InvalidOperation
from pathlib import Path

folder = tempfile.TemporaryDirectory()
ledger = Path(folder.name) / "ledger.csv"
ledger.write_text(
    "account,amount\n"
    "rent,-850.00\n"
    "salary,2400.50\n"
    "food,-12.30\n"
    "food,abc\n"
    "food,-45.10\n"
    "salary\n"
    "fun,-19.99\n",
    encoding="utf-8",
)
```

<details><summary><b>Hint</b></summary>

`Decimal("abc")` raises `InvalidOperation`: catch it inside `parse_row` and `raise LedgerError(...) from error`. Read all rows with `list(csv.reader(file))`, then `enumerate(rows[1:], start=2)` gives file line numbers. JSON can't store a `Decimal`, so convert with `str()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
line 5: bad amount 'abc'
line 7: expected 2 fields, got 1
food: -57.40
fun: -19.99
rent: -850.00
salary: 2400.50
net: 1473.11
summary.json holds 4 accounts
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import defaultdict


class LedgerError(Exception):
    pass


def parse_row(row):
    if len(row) != 2:
        raise LedgerError(f"expected 2 fields, got {len(row)}")
    account, text = row
    try:
        amount = Decimal(text)
    except InvalidOperation as error:
        raise LedgerError(f"bad amount {text!r}") from error
    return account, amount


with open(ledger, newline="", encoding="utf-8") as file:
    rows = list(csv.reader(file))

totals = defaultdict(Decimal)
for line_number, row in enumerate(rows[1:], start=2):
    try:
        account, amount = parse_row(row)
    except LedgerError as error:
        print(f"line {line_number}: {error}")
        continue
    totals[account] += amount

for account in sorted(totals):
    print(f"{account}: {totals[account]}")
print(f"net: {sum(totals.values())}")

summary_path = ledger.parent / "summary.json"
summary = {account: str(totals[account]) for account in sorted(totals)}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
loaded = json.loads(summary_path.read_text(encoding="utf-8"))
print(f"summary.json holds {len(loaded)} accounts")
folder.cleanup()
```

`Decimal` keeps money exact: `-12.30 + -45.10` is exactly `-57.40`, trailing zero included, where floats could give `-57.400000000000006`. `parse_row` turns two different low-level problems into one `LedgerError`, and `from error` keeps the original cause attached for debugging. The loop knows the line number, so *it* adds that to the report: each layer adds what it knows. `defaultdict(Decimal)` starts each account at `Decimal('0')`. Writing amounts as strings means reading them back with `Decimal(text)` loses nothing.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import defaultdict


class LedgerError(Exception):
    pass


def parse_row(row):
    match row:
        case [account, text]:
            try:
                return account.strip(), Decimal(text)
            except InvalidOperation as error:
                raise LedgerError(f"bad amount {text!r}") from error
        case _:
            raise LedgerError(f"expected 2 fields, got {len(row)}")


def load_totals(path):
    totals, problems = defaultdict(Decimal), []
    with path.open(newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        for line_number, row in enumerate(reader, start=2):
            try:
                account, amount = parse_row(row)
            except LedgerError as error:
                problems.append(f"line {line_number}: {error}")
            else:
                totals[account] += amount
    return dict(sorted(totals.items())), problems


totals, problems = load_totals(ledger)
print(*problems, sep="\n")
for account, amount in totals.items():
    print(f"{account}: {amount}")
print(f"net: {sum(totals.values(), Decimal(0))}")

summary_path = ledger.with_name("summary.json")
summary_path.write_text(json.dumps(totals, default=str, indent=2), encoding="utf-8")
print(f"summary.json holds {len(json.loads(summary_path.read_text(encoding='utf-8')))} accounts")
folder.cleanup()
```

`load_totals` returns the problems instead of printing them, so a caller could also write them to a log. Reading row by row with `next(reader)` (NB22) to skip the header works for files of any size, and `json.dumps(..., default=str)` converts the `Decimal`s on the way out. `try` / `except` / `else` keeps the success path out of the `try`, so only `parse_row` errors get caught.

</details>

In [ ]:
import csv
import json
import tempfile
from decimal import Decimal, InvalidOperation
from pathlib import Path

folder = tempfile.TemporaryDirectory()
ledger = Path(folder.name) / "ledger.csv"
ledger.write_text(
    "account,amount\n"
    "rent,-850.00\n"
    "salary,2400.50\n"
    "food,-12.30\n"
    "food,abc\n"
    "food,-45.10\n"
    "salary\n"
    "fun,-19.99\n",
    encoding="utf-8",
)

# your code here


### NB20-Q22 · L5 · Transactions with a stack · Write

Write a `Store`: an in-memory key-value database with **nested
transactions**. Commands:

- `SET key value`, `GET key` (prints the value, or `NULL`), `DELETE key`
- `BEGIN` starts a transaction (they can be nested)
- `ROLLBACK` undoes everything since the most recent `BEGIN`
- `COMMIT` keeps the changes of the most recent `BEGIN`

`ROLLBACK` or `COMMIT` with no open transaction, and an unknown command,
raise a custom `StoreError`; the loop prints `error: <message>`. Only `GET`
and errors print anything.

```text
20
NULL
20
20
error: no transaction
NULL
error: unknown command BOGUS
```

**Starting code (already in the cell below):**

```python
commands = ["SET a 10", "BEGIN", "SET a 20", "GET a", "BEGIN", "DELETE a", "GET a",
            "ROLLBACK", "GET a", "COMMIT", "GET a", "ROLLBACK", "GET b", "BOGUS"]
```

<details><summary><b>Hint</b></summary>

Keep a stack of snapshots. `BEGIN` pushes a **copy** of the data (not the data itself!), `ROLLBACK` pops a snapshot and restores it, `COMMIT` pops one and throws it away.

</details>

<details><summary><b>Expected output</b></summary>

```text
20
NULL
20
20
error: no transaction
NULL
error: unknown command BOGUS
```

</details>

<details><summary><b>Solution</b></summary>

```python
class StoreError(Exception):
    pass


class Store:
    def __init__(self):
        self.data = {}
        self.snapshots = []

    def run(self, command):
        parts = command.split()
        name = parts[0]
        if name == "SET":
            self.data[parts[1]] = parts[2]
        elif name == "GET":
            print(self.data.get(parts[1], "NULL"))
        elif name == "DELETE":
            self.data.pop(parts[1], None)
        elif name == "BEGIN":
            self.snapshots.append(dict(self.data))
        elif name == "ROLLBACK":
            if not self.snapshots:
                raise StoreError("no transaction")
            self.data = self.snapshots.pop()
        elif name == "COMMIT":
            if not self.snapshots:
                raise StoreError("no transaction")
            self.snapshots.pop()
        else:
            raise StoreError(f"unknown command {name}")


store = Store()
for command in commands:
    try:
        store.run(command)
    except StoreError as error:
        print("error:", error)
```

Nested transactions are last in, first out: the most recent `BEGIN` is the first one closed. So a stack of snapshots is the natural shape. `dict(self.data)` is the trap: pushing `self.data` itself would store a second name for the *same* dict, every later change would show up in the "snapshot" too, and rollback would undo nothing. `COMMIT` only drops the inner snapshot, so the outer transaction can still roll everything back.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class StoreError(Exception):
    pass


class Store:
    def __init__(self):
        self.data = {}
        self._undo_logs = []

    def _remember(self, key):
        if self._undo_logs and key not in self._undo_logs[-1]:
            self._undo_logs[-1][key] = self.data.get(key)

    def _close(self):
        if not self._undo_logs:
            raise StoreError("no transaction")
        return self._undo_logs.pop()

    def run(self, command):
        match command.split():
            case ["SET", key, value]:
                self._remember(key)
                self.data[key] = value
            case ["GET", key]:
                print(self.data.get(key, "NULL"))
            case ["DELETE", key]:
                self._remember(key)
                self.data.pop(key, None)
            case ["BEGIN"]:
                self._undo_logs.append({})
            case ["ROLLBACK"]:
                for key, old in self._close().items():
                    if old is None:
                        self.data.pop(key, None)
                    else:
                        self.data[key] = old
            case ["COMMIT"]:
                log = self._close()
                if self._undo_logs:
                    for key, old in log.items():
                        self._undo_logs[-1].setdefault(key, old)
            case [name, *_]:
                raise StoreError(f"unknown command {name}")


store = Store()
for command in commands:
    try:
        store.run(command)
    except StoreError as error:
        print(f"error: {error}")
```

Copying the whole dict on every `BEGIN` is O(size of the database). The senior stores an **undo log** instead: only the old values of keys that actually change. That's how real databases do it. On `COMMIT` the inner log merges into the outer one, so an outer `ROLLBACK` still works. `match` (NB18) also rejects `SET` with a missing value instead of crashing with `IndexError`.

</details>

In [ ]:
commands = ["SET a 10", "BEGIN", "SET a 20", "GET a", "BEGIN", "DELETE a", "GET a",
            "ROLLBACK", "GET a", "COMMIT", "GET a", "ROLLBACK", "GET b", "BOGUS"]

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB20-P · L5 · Mini-project: Library Management System · Write

Build a small library system that brings the whole intermediate tier
together.

**Classes.** `Book` (`isbn`, `title`, `author`, `borrowed_by` which is a
member name or `None`), `Member` (`name`, `limit`, default `2`) and
`Library`, which holds both in dicts and has `add_book`, `add_member`,
`borrow(name, isbn)` and `give_back(name, isbn)`.

**Exceptions.** A base `LibraryError`, and subclasses raised by
`borrow` / `give_back`:

- `BookNotFound`: `no book with ISBN 999`
- `AlreadyBorrowed`: `Dune is already borrowed by ana`
- `LimitReached`: `ana already has 2 books (limit 2)`
- `NotBorrowed`: `ben has not borrowed Emma`

**Persistence.** `save(path)` writes the library to JSON. A separate
function `load_library(path)` builds a **new** `Library` from that file.

**Program.** Add the `books` and `members`, run every event (printing
`ana borrowed Dune`, `ana returned Dune` or `error: <message>`), save,
print what was saved, load it into a new library and print a report from
the **loaded** copy: books by title, then members by name with their
books (or `-`).

```text
ana borrowed Dune
error: Dune is already borrowed by ana
ana borrowed Emma
error: ana already has 2 books (limit 2)
error: no book with ISBN 999
error: ben has not borrowed Emma
ana returned Dune
ben borrowed Dune
saved 3 books and 3 members
--- report (reloaded) ---
Dune (Herbert): ben
Emma (Austen): ana
Ulysses (Joyce): available
ana: Emma
ben: Dune
cara: -
```

**Starting code (already in the cell below):**

```python
import json
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
save_path = Path(folder.name) / "library.json"

books = [("111", "Dune", "Herbert"), ("222", "Emma", "Austen"), ("333", "Ulysses", "Joyce")]
members = ["ana", "ben", "cara"]
events = [
    ("borrow", "ana", "111"),
    ("borrow", "ben", "111"),
    ("borrow", "ana", "222"),
    ("borrow", "ana", "333"),
    ("borrow", "ben", "999"),
    ("return", "ben", "222"),
    ("return", "ana", "111"),
    ("borrow", "ben", "111"),
]
```

<details><summary><b>Hint</b></summary>

Decide where each fact lives. If `Book.borrowed_by` is the only record of who has what, a member's books can always be worked out from the books, and the two can never disagree. Save books and members as lists of plain dicts. Check everything in `borrow` before changing anything.

</details>

<details><summary><b>Expected output</b></summary>

```text
ana borrowed Dune
error: Dune is already borrowed by ana
ana borrowed Emma
error: ana already has 2 books (limit 2)
error: no book with ISBN 999
error: ben has not borrowed Emma
ana returned Dune
ben borrowed Dune
saved 3 books and 3 members
--- report (reloaded) ---
Dune (Herbert): ben
Emma (Austen): ana
Ulysses (Joyce): available
ana: Emma
ben: Dune
cara: -
```

</details>

<details><summary><b>Solution</b></summary>

```python
class LibraryError(Exception):
    pass


class BookNotFound(LibraryError):
    pass


class AlreadyBorrowed(LibraryError):
    pass


class LimitReached(LibraryError):
    pass


class NotBorrowed(LibraryError):
    pass


class Book:
    def __init__(self, isbn, title, author, borrowed_by=None):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.borrowed_by = borrowed_by


class Member:
    def __init__(self, name, limit=2):
        self.name = name
        self.limit = limit


class Library:
    def __init__(self):
        self.books = {}
        self.members = {}

    def add_book(self, book):
        self.books[book.isbn] = book

    def add_member(self, member):
        self.members[member.name] = member

    def find_book(self, isbn):
        if isbn not in self.books:
            raise BookNotFound(f"no book with ISBN {isbn}")
        return self.books[isbn]

    def books_of(self, name):
        return [book for book in self.books.values() if book.borrowed_by == name]

    def borrow(self, name, isbn):
        book = self.find_book(isbn)
        member = self.members[name]
        if book.borrowed_by is not None:
            raise AlreadyBorrowed(f"{book.title} is already borrowed by {book.borrowed_by}")
        if len(self.books_of(name)) >= member.limit:
            raise LimitReached(f"{name} already has {member.limit} books "
                               f"(limit {member.limit})")
        book.borrowed_by = name
        return book

    def give_back(self, name, isbn):
        book = self.find_book(isbn)
        if book.borrowed_by != name:
            raise NotBorrowed(f"{name} has not borrowed {book.title}")
        book.borrowed_by = None
        return book

    def save(self, path):
        data = {
            "books": [{"isbn": b.isbn, "title": b.title, "author": b.author,
                       "borrowed_by": b.borrowed_by} for b in self.books.values()],
            "members": [{"name": m.name, "limit": m.limit} for m in self.members.values()],
        }
        path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def load_library(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    library = Library()
    for row in data["books"]:
        library.add_book(Book(row["isbn"], row["title"], row["author"], row["borrowed_by"]))
    for row in data["members"]:
        library.add_member(Member(row["name"], row["limit"]))
    return library


library = Library()
for isbn, title, author in books:
    library.add_book(Book(isbn, title, author))
for name in members:
    library.add_member(Member(name))

for action, name, isbn in events:
    try:
        if action == "borrow":
            book = library.borrow(name, isbn)
            print(f"{name} borrowed {book.title}")
        else:
            book = library.give_back(name, isbn)
            print(f"{name} returned {book.title}")
    except LibraryError as error:
        print("error:", error)

library.save(save_path)
print(f"saved {len(library.books)} books and {len(library.members)} members")

reloaded = load_library(save_path)
print("--- report (reloaded) ---")
for book in sorted(reloaded.books.values(), key=lambda b: b.title):
    print(f"{book.title} ({book.author}): {book.borrowed_by or 'available'}")
for name in sorted(reloaded.members):
    titles = sorted(book.title for book in reloaded.books_of(name))
    print(f"{name}: {', '.join(titles) or '-'}")
folder.cleanup()
```

The key design decision: **who has which book is stored in one place**, `Book.borrowed_by`. `books_of(name)` works it out when needed, so a member's list and the books can never disagree, and the JSON file needs no duplicate data. `borrow` runs every check before its single change, so a failed borrow changes nothing. The exception classes form a small hierarchy: the loop catches `LibraryError` and gets all four, while other bugs (a typo, a `KeyError`) still surface loudly. Reporting from the *reloaded* copy proves the save and load round trip really works.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass, asdict


class LibraryError(Exception):
    pass


class BookNotFound(LibraryError):
    pass


class AlreadyBorrowed(LibraryError):
    pass


class LimitReached(LibraryError):
    pass


class NotBorrowed(LibraryError):
    pass


@dataclass
class Book:
    isbn: str
    title: str
    author: str
    borrowed_by: str | None = None


@dataclass
class Member:
    name: str
    limit: int = 2


class Library:
    def __init__(self, books=(), members=()):
        self.books = {book.isbn: book for book in books}
        self.members = {member.name: member for member in members}

    @classmethod
    def load(cls, path):
        data = json.loads(path.read_text(encoding="utf-8"))
        return cls(books=[Book(**row) for row in data["books"]],
                   members=[Member(**row) for row in data["members"]])

    def save(self, path):
        data = {"books": [asdict(b) for b in self.books.values()],
                "members": [asdict(m) for m in self.members.values()]}
        path.write_text(json.dumps(data, indent=2), encoding="utf-8")

    def _book(self, isbn):
        try:
            return self.books[isbn]
        except KeyError:
            raise BookNotFound(f"no book with ISBN {isbn}") from None

    def books_of(self, name):
        return sorted((b for b in self.books.values() if b.borrowed_by == name),
                      key=lambda b: b.title)

    def borrow(self, name, isbn):
        book, member = self._book(isbn), self.members[name]
        if book.borrowed_by is not None:
            raise AlreadyBorrowed(f"{book.title} is already borrowed by {book.borrowed_by}")
        if len(self.books_of(name)) >= member.limit:
            raise LimitReached(f"{name} already has {member.limit} books (limit {member.limit})")
        book.borrowed_by = name
        return book

    def give_back(self, name, isbn):
        book = self._book(isbn)
        if book.borrowed_by != name:
            raise NotBorrowed(f"{name} has not borrowed {book.title}")
        book.borrowed_by = None
        return book

    def report(self):
        for book in sorted(self.books.values(), key=lambda b: b.title):
            yield f"{book.title} ({book.author}): {book.borrowed_by or 'available'}"
        for name in sorted(self.members):
            yield f"{name}: {', '.join(b.title for b in self.books_of(name)) or '-'}"


library = Library(books=[Book(*row) for row in books], members=[Member(n) for n in members])
ACTIONS = {"borrow": (library.borrow, "borrowed"), "return": (library.give_back, "returned")}
for action, name, isbn in events:
    method, verb = ACTIONS[action]
    try:
        print(f"{name} {verb} {method(name, isbn).title}")
    except LibraryError as error:
        print(f"error: {error}")

library.save(save_path)
print(f"saved {len(library.books)} books and {len(library.members)} members")
print("--- report (reloaded) ---")
print("\n".join(Library.load(save_path).report()))
folder.cleanup()
```

`@dataclass` (NB21) writes `__init__` and `__repr__`, and `asdict` / `Book(**row)` make saving and loading one line each. A `classmethod` (NB21) `Library.load` is the standard "alternative constructor". `report()` *yields* lines (NB22) instead of printing, so the same report could go to a file or a web page. `raise ... from None` hides the internal `KeyError` the user doesn't care about. The design decision is the same as the beginner's: one source of truth for who has which book.

</details>

In [ ]:
import json
import tempfile
from pathlib import Path

folder = tempfile.TemporaryDirectory()
save_path = Path(folder.name) / "library.json"

books = [("111", "Dune", "Herbert"), ("222", "Emma", "Austen"), ("333", "Ulysses", "Joyce")]
members = ["ana", "ben", "cara"]
events = [
    ("borrow", "ana", "111"),
    ("borrow", "ben", "111"),
    ("borrow", "ana", "222"),
    ("borrow", "ana", "333"),
    ("borrow", "ben", "999"),
    ("return", "ben", "222"),
    ("return", "ana", "111"),
    ("borrow", "ben", "111"),
]

# your code here


---

## Done

Next up: **NB21 · OOP Advanced** in `advanced/`.